# E3 — Jeu d’évaluation commun aux quatre horizons

**Objectif :** conserver exclusivement les couples `(uuid, timestamp, split)` présents aux quatre horizons **et dans le protocole original à 15 minutes**. Les cibles restent spécifiques à chaque horizon. Les modèles peuvent être entraînés sur leurs partitions complètes, mais les évaluations comparatives doivent utiliser les indices de test communs produits ici.

Ce notebook **ne modifie aucun fichier existant**. Il vérifie les clés, l’ordre des tenseurs, les valeurs des cibles et les correspondances temporelles. Les fichiers de sortie sont placés dans `outputs/E3_common_evaluation/`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import json

ROOT = Path.cwd().resolve()
E3 = ROOT / 'outputs/E3_aligned_representations'
OUT = ROOT / 'outputs/E3_common_evaluation'
OUT.mkdir(parents=True, exist_ok=True)
HORIZONS = (15, 30, 60, 120)
SPLITS = ('train', 'validation', 'test')

# Renseigner le chemin réel du fichier original si nécessaire.
ORIGINAL_KEYS = ROOT / 'outputs/07D_final_alignment/01_final_common_sample_keys.csv'
if not ORIGINAL_KEYS.exists():
    matches = [p for p in ROOT.rglob('01_final_common_sample_keys.csv') if 'E3_common_evaluation' not in str(p)]
    if len(matches) != 1:
        raise FileNotFoundError(f'Fichier de référence introuvable ou ambigu : {matches}. Définir ORIGINAL_KEYS manuellement.')
    ORIGINAL_KEYS = matches[0]
print('Reference:', ORIGINAL_KEYS)
print('E3:', E3)


In [ ]:
KEYS = ['uuid', 'timestamp', 'split']
def normalize_keys(df, label):
    assert set(KEYS).issubset(df.columns), f'{label}: missing columns {set(KEYS)-set(df.columns)}'
    df = df.copy()
    df['uuid'] = df['uuid'].astype(str).str.strip()
    df['split'] = df['split'].astype(str).str.strip().str.lower()
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='raise')
    assert df['timestamp'].notna().all(), f'{label}: null timestamps'
    assert df['split'].isin(SPLITS).all(), f'{label}: unknown splits'
    assert not df.duplicated(KEYS).any(), f'{label}: duplicate keys'
    return df

orig = normalize_keys(pd.read_csv(ORIGINAL_KEYS), 'Original')
assert len(orig)==92316, f'Original reference count changed: {len(orig)}'
meta={}
for h in HORIZONS:
    p=E3/f'h{h:03d}'/'04_common_node_keys.csv'
    assert p.exists(), p
    d=normalize_keys(pd.read_csv(p), f'h{h}')
    assert 'target_timestamp' in d and 'target' in d
    d['target_timestamp']=pd.to_datetime(d['target_timestamp'],utc=True)
    assert (d.target_timestamp-d.timestamp).eq(pd.Timedelta(minutes=h)).all(), f'h{h}: incorrect target offset'
    assert d['node_index'].between(0,6).all()
    assert len(d)%7==0
    meta[h]=d
    print(f'{h:3d} min: {len(d):,} rows')

# Join all four sets to the original keys: this is an intersection, not a union.
common=orig[KEYS].copy()
for h in HORIZONS:
    common=common.merge(meta[h][KEYS],on=KEYS,how='inner',validate='one_to_one')
assert len(common)>0, 'Empty intersection'
assert common.groupby('split').size().gt(0).all(), 'One split missing'
assert common.groupby(['split','timestamp']).size().eq(7).all(), 'Incomplete seven-device snapshot in intersection'
common=common.sort_values(['split','timestamp','uuid']).reset_index(drop=True)
common.to_csv(OUT/'01_common_keys_all_horizons.csv',index=False)
print('Intersection:',len(common),'rows; snapshots:',len(common)//7)
print(common.groupby('split').size().to_string())


In [ ]:
# Per-horizon indices refer to original E3 tensor order, NOT to sorted common keys.
# The indices can be applied to each horizon's existing predictions without retraining.
rows=[]
for h in HORIZONS:
    d=meta[h]
    for split in SPLITS:
        part=d.loc[d.split.eq(split)].copy().reset_index(drop=True)
        wanted=common.loc[common.split.eq(split),KEYS]
        # The tensor metadata for each split has the same ordering as 04_common_node_keys.csv filtered by split.
        part['_row_idx']=np.arange(len(part),dtype=np.int64)
        selected=part.merge(wanted,on=KEYS,how='inner',validate='one_to_one')
        assert len(selected)==len(wanted), f'h{h} {split}: key mismatch'
        # Stable order: timestamp, node_index (same order for all horizons).
        selected=selected.sort_values(['timestamp','node_index']).reset_index(drop=True)
        idx=selected['_row_idx'].to_numpy(dtype=np.int64)
        assert len(np.unique(idx))==len(idx)
        np.save(OUT/f'indices_h{h:03d}_{split}.npy',idx)
        selected.drop(columns=['_row_idx']).to_csv(OUT/f'keys_targets_h{h:03d}_{split}.csv',index=False)
        # Confirm the target values correspond exactly to the tensor at these indices.
        y=np.load(E3/f'h{h:03d}'/f'y_sequence_{split}.npy',mmap_mode='r')
        assert len(y)==len(part), f'h{h} {split}: tensor length mismatch'
        assert np.allclose(np.asarray(y[idx]),selected['target'].to_numpy(),rtol=1e-5,atol=1e-4), f'h{h} {split}: target/tensor mismatch'
        rows.append({'horizon_min':h,'split':split,'original_horizon_samples':len(part),'common_samples':len(idx),'excluded_for_comparison':len(part)-len(idx)})

audit=pd.DataFrame(rows)
audit.to_csv(OUT/'02_common_evaluation_audit.csv',index=False)
print(audit.to_string(index=False))


In [ ]:
# Verify identical device-time-split keys and identical historical inputs across horizons.
for split in SPLITS:
    keys_ref=None; x_ref=None
    for h in HORIZONS:
        d=pd.read_csv(OUT/f'keys_targets_h{h:03d}_{split}.csv')
        keys=list(zip(d['uuid'].astype(str),pd.to_datetime(d['timestamp'],utc=True),d['split']))
        if keys_ref is None: keys_ref=keys
        else: assert keys==keys_ref, f'Key ordering mismatch at h{h}, {split}'
        idx=np.load(OUT/f'indices_h{h:03d}_{split}.npy')
        x=np.load(E3/f'h{h:03d}'/f'X_sequence_{split}_unscaled.npy',mmap_mode='r')
        xi=np.asarray(x[idx])
        if x_ref is None: x_ref=xi
        else: assert np.array_equal(xi,x_ref), f'Input history mismatch at h{h}, {split}'
    print(f'{split}: identical keys and historical inputs across all horizons ({len(keys_ref)} samples)')

(OUT/'03_protocol.json').write_text(json.dumps({
  'horizons_minutes':list(HORIZONS),'source':'E3_aligned_representations',
  'reference_keys':str(ORIGINAL_KEYS),'common_key_columns':KEYS,
  'common_evaluation':'intersection of original 15-min keys and all four E3 horizons',
  'training_policy':'train on full horizon-specific training sets; compare on common test indices',
  'warning':'Predictions must be indexed in the exact order of metadata_{split}.csv; no inference of causality or temporal leakage from this check alone.'
},indent=2),encoding='utf-8')
print('Saved outputs to:',OUT)


## Comment utiliser les indices

Pour un modèle évalué à l’horizon `h`, appliquer `indices_hXXX_test.npy` aux prédictions **dans l’ordre exact des lignes de** `metadata_test.csv` / `y_sequence_test.npy`. Pour un modèle graphique, aplatir les prédictions `[snapshots, 7]` avant l’indexation, en conservant l’ordre des nœuds.

**Important :** ce notebook construit un jeu d’évaluation commun, pas un nouveau jeu d’entraînement. Il ne vérifie pas la résolution des capteurs ni les effets des partitions glissantes. Il n'exécute aucun modèle.
